In [ ]:

import os
import copy
import json
import random
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
import matplotlib.pyplot as plt


torch.set_default_device("cpu")

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


FLOW_PATH = r"/root/autodl-fs/加州流量24年.csv"
SPEED_PATH = r"/root/autodl-fs/加州速度24年.csv"
SENSOR_PATH = r"/root/autodl-fs/加州24年传感器_补充后.csv"

DATA_DIR = r"/root/autodl-fs/preprocess_result24年"
RESULT_ROOT = r"/root/autodl-fs/full_model_results24年"
SENS_ROOT = os.path.join(RESULT_ROOT, "sensitivity_analysis")
os.makedirs(SENS_ROOT, exist_ok=True)

INPUT_LEN = 12
PRED_LEN = 6
TEST_RATIO = 0.20
VAL_RATIO = 0.15

TOP_K = 8
SIGMA_KM = 2.0

EPOCHS = 80
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 12

D_MODEL = 64
GNN_LAYERS = 2
NHEAD = 4
TRANS_LAYERS = 2
DIM_FEEDFORWARD = 128
DROPOUT = 0.1

BATCH_SIZE = 64
NUM_WORKERS = 0
BASE_SEED = 42

torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前设备：{DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU名称：{torch.cuda.get_device_name(0)}")
    print(f"GPU显存：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


FULL_MODEL_CFG = {
    "name": "full_model_all_features",
    "seed": BASE_SEED,
    "use_speed": True,
    "use_time": True,
    "use_spatial": True,
    "graph_modes": ["distance", "flow_corr", "speed_corr", "region"],
    "fusion_mode": "learnable",
    "d_model": D_MODEL,
    "gnn_layers": GNN_LAYERS,
    "nhead": NHEAD,
    "trans_layers": TRANS_LAYERS,
    "dim_feedforward": DIM_FEEDFORWARD,
    "dropout": DROPOUT,
}


class TrafficWindowDataset(Dataset):
    def __init__(self, X, y, input_len=INPUT_LEN, pred_len=PRED_LEN):
        self.X = X
        self.y = y
        self.input_len = input_len
        self.pred_len = pred_len
        self.T = X.shape[0]

    def __len__(self):
        return self.T - self.input_len - self.pred_len + 1

    def __getitem__(self, idx):
        x = self.X[idx: idx + self.input_len]
        y = self.y[idx + self.input_len: idx + self.input_len + self.pred_len]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# ===================== 载入预处理结果 =====================
def load_preprocessed_data():
    data = np.load(os.path.join(DATA_DIR, "dataset24年.npz"))

    X_train = data["X_train"].astype(np.float32)
    X_val = data["X_val"].astype(np.float32)
    X_test = data["X_test"].astype(np.float32)
    y_train = data["y_train"].astype(np.float32)
    y_val = data["y_val"].astype(np.float32)
    y_test = data["y_test"].astype(np.float32)

    with open(os.path.join(DATA_DIR, "scalers24年.pkl"), "rb") as f:
        scaler_dict = pickle.load(f)

    y_scaler = scaler_dict["y_scaler"]
    common_sensors = pd.read_csv(os.path.join(DATA_DIR, "common_sensors24年.csv"))["sensor_id"].tolist()

    return X_train, X_val, X_test, y_train, y_val, y_test, y_scaler, common_sensors

def load_graphs(graph_modes=None):
    g = np.load(os.path.join(DATA_DIR, "graph_mats24年.npz"))
    graph_map = {
        "distance": torch.tensor(g["A_distance"], dtype=torch.float32, device=DEVICE),
        "flow_corr": torch.tensor(g["A_flow_corr"], dtype=torch.float32, device=DEVICE),
        "speed_corr": torch.tensor(g["A_speed_corr"], dtype=torch.float32, device=DEVICE),
        "region": torch.tensor(g["A_region"], dtype=torch.float32, device=DEVICE),
    }

    if graph_modes is None:
        graph_modes = ["distance", "flow_corr", "speed_corr", "region"]

    A_list = [graph_map[m] for m in graph_modes if m in graph_map]
    return A_list

# ===================== 特征mask =====================
def apply_feature_mask(x, cfg):
    x = x.clone()
    if not cfg.get("use_speed", True):
        x[..., 1] = 0.0
    if not cfg.get("use_time", True):
        x[..., 2:] = 0.0
    return x

# ===================== 模型结构 =====================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class MultiGraphConv(nn.Module):
    def __init__(self, in_dim, out_dim, num_graphs=4, dropout=0.1, fusion_mode="learnable"):
        super().__init__()
        self.num_graphs = num_graphs
        self.fusion_mode = fusion_mode
        self.proj = nn.ModuleList([nn.Linear(in_dim, out_dim) for _ in range(num_graphs)])
        if fusion_mode == "learnable":
            self.graph_logits = nn.Parameter(torch.zeros(num_graphs))
        else:
            self.register_parameter("graph_logits", None)
        self.dropout = nn.Dropout(dropout)
        self.residual = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x, A_list):
        B, T, N, D = x.shape
        x_bt = x.reshape(B * T, N, D)

        outs = []
        for i, A in enumerate(A_list):
            agg = torch.einsum("ij,bjd->bid", A, x_bt)
            out = F.relu(self.proj[i](agg))
            outs.append(out)

        if self.fusion_mode == "learnable":
            weights = torch.softmax(self.graph_logits, dim=0)
        else:
            weights = torch.ones(self.num_graphs, device=x.device) / max(self.num_graphs, 1)

        fused = 0.0
        for i in range(self.num_graphs):
            fused = fused + weights[i] * outs[i]

        fused = self.dropout(fused)
        res = self.residual(x_bt)
        fused = self.norm(fused + res)

        return fused.reshape(B, T, N, -1)

class TemporalSeq2SeqTransformer(nn.Module):
    def __init__(self, d_model, pred_len, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.pred_len = pred_len
        self.pos_enc = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        self.future_queries = nn.Parameter(torch.randn(pred_len, d_model) * 0.02)
        self.out_proj = nn.Linear(d_model, 1)

    def forward(self, x):
        B, T, N, D = x.shape
        x = x.permute(0, 2, 1, 3).contiguous().reshape(B * N, T, D)

        memory = self.encoder(self.pos_enc(x))

        tgt = self.future_queries.unsqueeze(0).expand(B * N, -1, -1)
        tgt = self.pos_enc(tgt)
        dec = self.decoder(tgt=tgt, memory=memory)

        y = self.out_proj(dec).squeeze(-1)
        y = y.reshape(B, N, self.pred_len).permute(0, 2, 1).contiguous()
        return y

class MGSTTransformer(nn.Module):
    def __init__(self,
                 in_dim,
                 num_nodes,
                 d_model=64,
                 gnn_layers=2,
                 nhead=4,
                 num_layers=2,
                 dim_feedforward=128,
                 dropout=0.1,
                 pred_len=6,
                 num_graphs=4,
                 use_spatial=True,
                 fusion_mode="learnable"):
        super().__init__()
        self.use_spatial = use_spatial
        self.input_proj = nn.Linear(in_dim, d_model)

        if self.use_spatial:
            self.spatial_layers = nn.ModuleList([
                MultiGraphConv(d_model, d_model, num_graphs=num_graphs, dropout=dropout, fusion_mode=fusion_mode)
                for _ in range(gnn_layers)
            ])
        else:
            self.spatial_layers = nn.ModuleList([])

        self.temporal_predictor = TemporalSeq2SeqTransformer(
            d_model=d_model,
            pred_len=pred_len,
            nhead=nhead,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )

    def forward(self, x, A_list):
        x = self.input_proj(x)
        if self.use_spatial and len(A_list) > 0:
            for gnn in self.spatial_layers:
                x = gnn(x, A_list)
        out = self.temporal_predictor(x)
        return out

# ===================== 指标 =====================
def inverse_transform_3d(scaler, arr):
    shape = arr.shape
    arr2 = arr.reshape(-1, 1)
    inv = scaler.inverse_transform(arr2).reshape(shape)
    return inv

def safe_mape(y_true, y_pred, eps=1e-8):
    mask = np.abs(y_true) > eps
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0

def smape(y_true, y_pred, eps=1e-8):
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0

def wape(y_true, y_pred, eps=1e-8):
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps) * 100.0

def compute_metrics(y_true, y_pred):
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)

    mse = mean_squared_error(yt, yp)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(yt, yp)
    mape = safe_mape(yt, yp)
    smape_v = smape(yt, yp)
    r2 = r2_score(yt, yp)
    evs = explained_variance_score(yt, yp)
    wape_v = wape(yt, yp)

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE(%)": mape,
        "sMAPE(%)": smape_v,
        "R2": r2,
        "EVS": evs,
        "WAPE(%)": wape_v
    }


def evaluate_model(model, loader, A_list, y_scaler, device, cfg):
    model.eval()
    preds_all = []
    trues_all = []

    with torch.no_grad():
        for x, y in loader:
            x = apply_feature_mask(x, cfg).to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
                pred = model(x, A_list)

            preds_all.append(pred.detach().cpu().numpy())
            trues_all.append(y.detach().cpu().numpy())

    preds = np.concatenate(preds_all, axis=0)
    trues = np.concatenate(trues_all, axis=0)

    preds_inv = inverse_transform_3d(y_scaler, preds)
    trues_inv = inverse_transform_3d(y_scaler, trues)

    overall_metrics = compute_metrics(trues_inv, preds_inv)
    horizon_metrics = {}
    for h in range(preds_inv.shape[1]):
        horizon_metrics[f"H{h + 1}"] = compute_metrics(trues_inv[:, h, :], preds_inv[:, h, :])

    return overall_metrics, horizon_metrics, preds_inv, trues_inv

# ===================== 训练 =====================
def train_model(model, train_loader, val_loader, A_list, epochs, lr, weight_decay, patience, device, cfg):
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    model = model.to(device)

    best_val = float("inf")
    best_state = None
    bad_count = 0

    train_losses = []
    val_losses = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_train = 0.0
        n_train = 0

        for x, y in train_loader:
            x = apply_feature_mask(x, cfg).to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
                pred = model(x, A_list)
                loss = criterion(pred, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_train += loss.item()
            n_train += 1

        train_loss = total_train / max(n_train, 1)
        train_losses.append(train_loss)

        model.eval()
        total_val = 0.0
        n_val = 0

        with torch.no_grad():
            for x, y in val_loader:
                x = apply_feature_mask(x, cfg).to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
                    pred = model(x, A_list)
                    loss = criterion(pred, y)

                total_val += loss.item()
                n_val += 1

        val_loss = total_val / max(n_val, 1)
        val_losses.append(val_loss)

        print(f"Epoch [{epoch:03d}/{epochs}] | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            bad_count = 0
        else:
            bad_count += 1
            if bad_count >= patience:
                print(f"早停触发：Epoch={epoch}, 最佳验证损失={best_val:.6f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, train_losses, val_losses

# ===================== DataLoader =====================
def create_dataloaders(X_train, X_val, X_test, y_train, y_val, y_test, batch_size=64, num_workers=0, seed=42):
    train_ds = TrafficWindowDataset(X_train, y_train)
    val_ds = TrafficWindowDataset(X_val, y_val)
    test_ds = TrafficWindowDataset(X_test, y_test)

    use_pin = torch.cuda.is_available()
    persistent = num_workers > 0
    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, pin_memory=use_pin,
        num_workers=num_workers, drop_last=True, persistent_workers=persistent,
        worker_init_fn=seed_worker if num_workers > 0 else None, generator=g
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, pin_memory=use_pin,
        num_workers=num_workers, persistent_workers=persistent,
        worker_init_fn=seed_worker if num_workers > 0 else None
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, pin_memory=use_pin,
        num_workers=num_workers, persistent_workers=persistent,
        worker_init_fn=seed_worker if num_workers > 0 else None
    )
    print(f"DataLoader：train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")
    return train_loader, val_loader, test_loader

# ===================== 敏感性分析配置 =====================
SENSITIVITY_SPACE = {
    "d_model": [32, 64, 128],
    "gnn_layers": [1, 2, 3],
    "nhead": [2, 4, 8],
    "trans_layers": [1, 2, 3],
    "dropout": [0.0, 0.1, 0.2, 0.3],
    "graph_modes": [
        ["distance"],
        ["distance", "flow_corr"],
        ["distance", "flow_corr", "speed_corr"],
        ["distance", "flow_corr", "speed_corr", "region"]
    ]
}

def _display_value(v):
    if isinstance(v, (list, tuple)):
        return "+".join(map(str, v))
    if isinstance(v, float):
        return f"{v:.2f}"
    return str(v)

def build_model_from_cfg(cfg, input_dim, num_nodes):
    A_list = load_graphs(cfg["graph_modes"])
    model = MGSTTransformer(
        in_dim=input_dim,
        num_nodes=num_nodes,
        d_model=cfg["d_model"],
        gnn_layers=cfg["gnn_layers"],
        nhead=cfg["nhead"],
        num_layers=cfg["trans_layers"],
        dim_feedforward=cfg["dim_feedforward"],
        dropout=cfg["dropout"],
        pred_len=PRED_LEN,
        num_graphs=len(A_list) if cfg["use_spatial"] else 0,
        use_spatial=cfg["use_spatial"],
        fusion_mode=cfg["fusion_mode"]
    )
    return model, A_list

def run_one_setting(param_name, param_value, data_cache, save_pred=False):
    cfg = copy.deepcopy(FULL_MODEL_CFG)
    cfg["seed"] = BASE_SEED
    cfg[param_name] = param_value
    cfg["name"] = f"sens_{param_name}_{_display_value(param_value).replace(' ', '').replace('/', '_')}"

    exp_dir = os.path.join(SENS_ROOT, cfg["name"])
    os.makedirs(exp_dir, exist_ok=True)

    set_seed(cfg["seed"])
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    X_train = data_cache["X_train"]
    X_val = data_cache["X_val"]
    X_test = data_cache["X_test"]
    y_train = data_cache["y_train"]
    y_val = data_cache["y_val"]
    y_test = data_cache["y_test"]
    y_scaler = data_cache["y_scaler"]
    common_sensors = data_cache["common_sensors"]

    print("\n" + "=" * 90)
    print(f"开始敏感性实验：{param_name} = {_display_value(param_value)}")
    print("=" * 90)

    train_loader, val_loader, test_loader = create_dataloaders(
        X_train, X_val, X_test, y_train, y_val, y_test,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, seed=cfg["seed"]
    )

    model, A_list = build_model_from_cfg(cfg, input_dim=X_train.shape[-1], num_nodes=len(common_sensors))
    model = model.to(DEVICE)

    print(f"模型设备：{next(model.parameters()).device}")
    print(f"图数量：{len(A_list)}")

    model, train_losses, val_losses = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        A_list=A_list,
        epochs=EPOCHS,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        patience=PATIENCE,
        device=DEVICE,
        cfg=cfg
    )

    overall_metrics, horizon_metrics, preds_inv, trues_inv = evaluate_model(
        model=model,
        loader=test_loader,
        A_list=A_list,
        y_scaler=y_scaler,
        device=DEVICE,
        cfg=cfg
    )

    result_row = {
        "param_name": param_name,
        "param_value": _display_value(param_value),
        "best_val_mse": float(np.min(val_losses)) if len(val_losses) > 0 else np.nan,
        "best_epoch": int(np.argmin(val_losses) + 1) if len(val_losses) > 0 else np.nan,
        **overall_metrics
    }

    pd.DataFrame([result_row]).to_csv(
        os.path.join(exp_dir, "summary.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    pd.DataFrame({
        "Epoch": np.arange(1, len(train_losses) + 1),
        "Train_MSE": train_losses,
        "Val_MSE": val_losses
    }).to_csv(
        os.path.join(exp_dir, "loss_curve.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    if save_pred:
        np.savez_compressed(
            os.path.join(exp_dir, "test_predictions.npz"),
            preds=preds_inv,
            trues=trues_inv
        )

    with open(os.path.join(exp_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)

    print(f"完成：{param_name} = {_display_value(param_value)}")
    print(f"RMSE={overall_metrics['RMSE']:.6f}, MAE={overall_metrics['MAE']:.6f}, MAPE={overall_metrics['MAPE(%)']:.4f}%")

    return result_row

def plot_sensitivity_curve(summary_df, param_name, save_dir):
    df = summary_df[summary_df["param_name"] == param_name].copy()
    if df.empty:
        return

    df = df.reset_index(drop=True)
    x = np.arange(len(df))
    x_labels = df["param_value"].astype(str).tolist()

    metrics_to_plot = ["RMSE", "MAE", "MAPE(%)"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), dpi=150)

    for ax, metric in zip(axes, metrics_to_plot):
        ax.plot(x, df[metric].values, marker="o", linewidth=2)
        ax.set_xlabel(param_name)
        ax.set_ylabel(metric)
        ax.set_title(f"{param_name} vs {metric}")
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=30, ha="right")

    fig.tight_layout()
    fig_path = os.path.join(save_dir, f"{param_name}_sensitivity.png")
    fig.savefig(fig_path, bbox_inches="tight")
    plt.close(fig)
    print(f"图已保存：{fig_path}")

def run_sensitivity_analysis():
    set_seed(BASE_SEED)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    X_train, X_val, X_test, y_train, y_val, y_test, y_scaler, common_sensors = load_preprocessed_data()
    data_cache = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "y_scaler": y_scaler,
        "common_sensors": common_sensors
    }

    all_rows = []
    study_order = ["d_model", "gnn_layers", "nhead", "trans_layers", "dropout", "graph_modes"]

    for param_name in study_order:
        print("\n" + "#" * 90)
        print(f"开始分析参数：{param_name}")
        print("#" * 90)

        for v in SENSITIVITY_SPACE[param_name]:
            row = run_one_setting(param_name, v, data_cache, save_pred=False)
            all_rows.append(row)

    summary_df = pd.DataFrame(all_rows)
    summary_df.to_csv(
        os.path.join(SENS_ROOT, "sensitivity_summary.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    for param_name in study_order:
        plot_sensitivity_curve(summary_df, param_name, SENS_ROOT)

    best_rows = []
    for param_name in study_order:
        sub = summary_df[summary_df["param_name"] == param_name].copy()
        if sub.empty:
            continue
        best_idx = sub["RMSE"].idxmin()
        best_rows.append(summary_df.loc[best_idx].to_dict())

    best_df = pd.DataFrame(best_rows)
    best_df.to_csv(
        os.path.join(SENS_ROOT, "best_settings_by_param.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    print("\n敏感性分析完成。")
    print(f"总结果表：{os.path.join(SENS_ROOT, 'sensitivity_summary.csv')}")
    print(f"最优设置表：{os.path.join(SENS_ROOT, 'best_settings_by_param.csv')}")

def main():
    run_sensitivity_analysis()

if __name__ == "__main__":
    main()